In [10]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [11]:
# Using sample data for testing purposes
data_path = Path("../../data/sample_resources.csv")
agent_output_path = Path("output/recommendations.json")
output_dir = Path("output")

print(f"Data path: {data_path}")
print(f"Agent output path: {agent_output_path}")
print(f"Output directory: {output_dir}")


Data path: ..\..\data\sample_resources.csv
Agent output path: output\recommendations.json
Output directory: output


In [12]:
df = pd.read_csv(data_path)
print(f"Rows: {len(df)}")
display(df.head())

Rows: 6


,resource_id,team,environment,service,monthly_cost,cpu_utilization,memory_utilization,uptime_required,policy_locked,schedule_eligible,storage_tier_eligible
0,vm-prod-01,core-platform,prod,vm,420.0,18,31,24x7,False,False,False
1,vm-prod-02,payments,prod,vm,510.0,44,62,24x7,True,False,False
2,vm-dev-01,analytics,dev,vm,260.0,12,28,business-hours,False,True,False
3,vm-test-01,marketing,test,vm,180.0,16,24,business-hours,False,True,False
4,storage-01,data-eng,prod,storage,190.0,0,0,24x7,False,False,True


In [13]:
def to_bool(series):
    if series.dtype == bool:
        return series
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
        .fillna(False)
    )

In [14]:
df["monthly_cost"] = pd.to_numeric(df["monthly_cost"], errors="coerce")
df["cpu_utilization"] = pd.to_numeric(df["cpu_utilization"], errors="coerce")
df["memory_utilization"] = pd.to_numeric(df["memory_utilization"], errors="coerce")
df["policy_locked"] = to_bool(df["policy_locked"])
df["schedule_eligible"] = to_bool(df["schedule_eligible"])
df["storage_tier_eligible"] = to_bool(df["storage_tier_eligible"])

In [6]:
quality_checks = {
    "null_resource_id": int(df["resource_id"].isna().sum()),
    "null_monthly_cost": int(df["monthly_cost"].isna().sum()),
    "negative_monthly_cost": int((df["monthly_cost"] < 0).sum()),
    "duplicate_resource_id": int(df["resource_id"].duplicated().sum()),
}
print("\nData quality checks")
display(pd.DataFrame([quality_checks]))
if any(quality_checks.values()):
    print("Warning: data quality issues detected. Review before trusting baseline outputs.")
else:
    print("Data quality checks passed.")


Data quality checks


,null_resource_id,null_monthly_cost,negative_monthly_cost,duplicate_resource_id
0,0,0,0,0


Data quality checks passed.


In [15]:
def compute_baseline_recommendation(row):
    if row["policy_locked"]:
        return None
    # Extract relevant fields from the row
    service = str(row["service"]).lower()
    environment = str(row["environment"]).lower()
    uptime = str(row["uptime_required"]).lower()
    # Check for VM scheduling recommendation
    if (
        service == "vm"
        and environment != "prod"
        and bool(row["schedule_eligible"])
        and uptime != "24x7"
    ):
        factor = 0.35
        return {
            "resource_id": row["resource_id"],
            "team": row["team"],
            "environment": row["environment"],
            "service": row["service"],
            "action": "schedule_shutdown",
            "estimated_monthly_saving": round(row["monthly_cost"] * factor, 2),
            "rule_name": "nonprod_schedule_candidate",
        }
    # Check for VM rightsizing recommendation
    if (
        service == "vm"
        and row["cpu_utilization"] < 25
        and row["memory_utilization"] < 40
    ):
        factor = 0.18 if environment == "prod" else 0.15
        return {
            "resource_id": row["resource_id"],
            "team": row["team"],
            "environment": row["environment"],
            "service": row["service"],
            "action": "rightsize_vm",
            "estimated_monthly_saving": round(row["monthly_cost"] * factor, 2),
            "rule_name": "vm_low_utilization",
        }
    # Check for storage tiering recommendation
    if service == "storage" and bool(row["storage_tier_eligible"]):
        factor = 0.22
        return {
            "resource_id": row["resource_id"],
            "team": row["team"],
            "environment": row["environment"],
            "service": row["service"],
            "action": "move_to_cool_tier",
            "estimated_monthly_saving": round(row["monthly_cost"] * factor, 2),
            "rule_name": "storage_tiering_candidate",
        }
    # If no recommendation applies, return None
    return None

In [16]:
# Compute baseline recommendations
baseline_recommendations = [
    rec for rec in (compute_baseline_recommendation(row) for _, row in df.iterrows()) if rec
]
baseline_df = pd.DataFrame(baseline_recommendations)
# Save baseline recommendations to JSON
print("\nBaseline recommendations")
display(baseline_df.sort_values(["estimated_monthly_saving"], ascending=False))
print(f"Recommendation count: {len(baseline_df)}")
print(f"Estimated monthly saving: ${baseline_df['estimated_monthly_saving'].sum():.2f}")

# Compute KPIs
portfolio_monthly_cost = df["monthly_cost"].sum()
baseline_monthly_saving = baseline_df["estimated_monthly_saving"].sum()
savings_rate = (baseline_monthly_saving / portfolio_monthly_cost * 100) if portfolio_monthly_cost else 0

kpi = pd.DataFrame([
    {
        "portfolio_monthly_cost": round(portfolio_monthly_cost, 2),
        "baseline_monthly_saving": round(baseline_monthly_saving, 2),
        "baseline_savings_rate_pct": round(savings_rate, 2),
        "baseline_recommendation_count": int(len(baseline_df)),
    }
])


Baseline recommendations


,resource_id,team,environment,service,action,estimated_monthly_saving,rule_name
1,vm-dev-01,analytics,dev,vm,schedule_shutdown,91.0,nonprod_schedule_candidate
0,vm-prod-01,core-platform,prod,vm,rightsize_vm,75.6,vm_low_utilization
2,vm-test-01,marketing,test,vm,schedule_shutdown,63.0,nonprod_schedule_candidate
3,storage-01,data-eng,prod,storage,move_to_cool_tier,41.8,storage_tiering_candidate


Recommendation count: 4
Estimated monthly saving: $271.40


In [10]:
#Printing KPIs and savings breakdowns
print("\nPortfolio KPI")
display(kpi)

if not baseline_df.empty:
    print("\nSavings by action")
    display(
        baseline_df.groupby(["action"], as_index=False)["estimated_monthly_saving"].sum()
        .sort_values("estimated_monthly_saving", ascending=False)
    )
    print("\nSavings by environment")
    display(
        baseline_df.groupby(["environment"], as_index=False)["estimated_monthly_saving"].sum()
        .sort_values("estimated_monthly_saving", ascending=False)
    )
    print("\nSavings by team")
    display(
        baseline_df.groupby(["team"], as_index=False)["estimated_monthly_saving"].sum()
        .sort_values("estimated_monthly_saving", ascending=False)
    )


Portfolio KPI


,portfolio_monthly_cost,baseline_monthly_saving,baseline_savings_rate_pct,baseline_recommendation_count
0,1640.0,271.4,16.55,4



Savings by action


,action,estimated_monthly_saving
2,schedule_shutdown,154.0
1,rightsize_vm,75.6
0,move_to_cool_tier,41.8



Savings by environment


,environment,estimated_monthly_saving
1,prod,117.4
0,dev,91.0
2,test,63.0



Savings by team


,team,estimated_monthly_saving
0,analytics,91.0
1,core-platform,75.6
3,marketing,63.0
2,data-eng,41.8


In [ ]:
# Export baseline results in the same shape as the agent output.
baseline_output_dir = Path("output")
baseline_output_dir.mkdir(parents=True, exist_ok=True)
baseline_output_path = baseline_output_dir / "baseline_recommendations.json"

# Build a lookup to bring original resource fields into each recommendation row.
resource_lookup = {
    row["resource_id"]: row.to_dict()
    for _, row in df.iterrows()
}

def build_suggested_change(action: str, saving: float) -> dict:
    if action == "rightsize_vm":
        return {"rightsize_instance_type": True, "estimated_savings": round(float(saving), 2)}
    if action == "schedule_shutdown":
        return {"enable_schedule": True, "estimated_savings": round(float(saving), 2)}
    if action == "move_to_cool_tier":
        return {"tier_storage_downgrade": True, "estimated_savings": round(float(saving), 2)}
    return {"estimated_savings": round(float(saving), 2)}

recommendations = []
for rec in baseline_df.to_dict(orient="records"):
    resource = resource_lookup[rec["resource_id"]]
    recommendations.append(
        {
            "resource_id": rec["resource_id"],
            "team": rec["team"],
            "environment": rec["environment"],
            "service": rec["service"],
            "monthly_cost": f"{float(resource['monthly_cost']):.2f}",
            "cpu_utilization": str(resource["cpu_utilization"]),
            "memory_utilization": str(resource["memory_utilization"]),
            "uptime_required": str(resource["uptime_required"]),
            "policy_locked": str(resource["policy_locked"]).lower(),
            "schedule_eligible": str(resource["schedule_eligible"]).lower(),
            "storage_tier_eligible": str(resource["storage_tier_eligible"]).lower(),
            "suggested_change": build_suggested_change(
                rec["action"],
                rec["estimated_monthly_saving"],
            )
        }
    )

baseline_output = {
    "summary": "Low-risk cost optimisation opportunities identified",
    "recommendations": recommendations,
}

with baseline_output_path.open("w", encoding="utf-8") as file_handle:
    json.dump(baseline_output, file_handle, indent=2)

print(f"Baseline results saved to {baseline_output_path}")

Baseline results saved to output\baseline_recommendations.json
